In [ ]:
!nvidia-smi

In [ ]:
# Instalasi Library
!pip install -q unsloth

# Fine-Tuning Mistral 7B dengan QLoRA Menggunakan Unsloth

Notebook ini digunakan untuk melakukan fine-tuning model Mistral-7B-Instruct-v0.3 pada tugas klasifikasi aspek usability aplikasi DANA menggunakan pendekatan QLoRA.

Model dilatih menggunakan:
- Dataset train gabungan (manual + pseudo-label)
- Validation set untuk model selection
- PEFT LoRA
- Unsloth FastLanguageModel
- Quantization 4-bit

In [ ]:
import torch

print(torch.__version__)

from unsloth import FastLanguageModel

print("Unsloth berhasil diimport")

In [ ]:
# Import Library
import os
import json
import pandas as pd
import matplotlib.pyplot as plt

from datasets import Dataset

from huggingface_hub import notebook_login

from trl import SFTTrainer, SFTConfig

from peft import (
    LoraConfig
)

In [ ]:
# Login Hugging Face
notebook_login()

In [ ]:
# Load Dataset
train_manual_df = pd.read_csv(
    "/kaggle/input/datasets/cindyzakya/usability-datasets-v2/train_manual_ground_truth.csv"
)

train_pseudo_df = pd.read_csv(
    "/kaggle/input/datasets/cindyzakya/usability-datasets-v2/train_pseudolabel.csv"
)

validation_df = pd.read_csv(
    "/kaggle/input/datasets/cindyzakya/usability-datasets-v2/validation_ground_truth.csv"
)

In [ ]:
print("Shape:", train_manual_df.shape)
print("Columns:", train_manual_df.columns.tolist(), "\n")

print("Shape:", train_pseudo_df.shape)
print("Columns:", train_pseudo_df.columns.tolist(), "\n")

print("Shape:", validation_df.shape)
print("Columns:", validation_df.columns.tolist())

In [ ]:
train_manual_df = train_manual_df[
    [
        "cleaned_content",
        "accuracy",
        "completeness",
        "satisfaction"
    ]
]

train_pseudo_df = train_pseudo_df[
    [
        "cleaned_content",
        "accuracy",
        "completeness",
        "satisfaction"
    ]
]

validation_df = validation_df[
    [
        "cleaned_content",
        "accuracy",
        "completeness",
        "satisfaction"
    ]
]

In [ ]:
train_df = pd.concat(
    [
        train_manual_df,
        train_pseudo_df
    ],
    ignore_index=True
)

In [ ]:
print("Train Set :", train_df.shape)
print("Validation Set :", validation_df.shape)

print("\nMissing Values")

print(
    train_df[
        [
            "cleaned_content",
            "accuracy",
            "completeness",
            "satisfaction"
        ]
    ].isna().sum()
)

In [ ]:
train_df[
    [
        "accuracy",
        "completeness",
        "satisfaction"
    ]
].sum()

## Memuat Model dan Tokenizer

Model Mistral-7B-Instruct-v0.3 dimuat menggunakan Unsloth dengan quantization 4-bit untuk mengurangi kebutuhan memori selama proses fine-tuning.

In [ ]:
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Mistral-7B-Instruct-v0.3",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

## Konfigurasi LoRA

Teknik Low-Rank Adaptation (LoRA) digunakan untuk melakukan fine-tuning secara efisien tanpa memperbarui seluruh parameter model.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    bias="none",

    use_gradient_checkpointing="unsloth",

    random_state=42,
)

## Format Data Pelatihan

Data pelatihan diformat menjadi pasangan percakapan user-assistant menggunakan chat template bawaan Mistral Instruct.

In [ ]:
def create_chat_example(row):

    review = row["cleaned_content"]

    label = json.dumps(
        {
            "accuracy": int(row["accuracy"]),
            "completeness": int(row["completeness"]),
            "satisfaction": int(row["satisfaction"])
        },
        ensure_ascii=False
    )

    messages = [
        {
            "role": "user",
            "content": review
        },
        {
            "role": "assistant",
            "content": label
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )

In [ ]:
train_df["text"] = train_df.apply(
    create_chat_example,
    axis=1
)

validation_df["text"] = validation_df.apply(
    create_chat_example,
    axis=1
)

In [ ]:
print(train_df["text"].iloc[0])

In [ ]:
train_dataset = Dataset.from_pandas(
    train_df[["text"]]
)

validation_dataset = Dataset.from_pandas(
    validation_df[["text"]]
)

In [ ]:
train_dataset

In [ ]:
validation_dataset

In [ ]:
# Training Configuration
training_args =  SFTConfig(
    output_dir="./outputs",

    num_train_epochs=3,

    learning_rate=2e-4,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=4,

    eval_strategy="steps",
    eval_steps=200,

    save_strategy="steps",
    save_steps=200,

    save_total_limit=3,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",
    greater_is_better=False,

    logging_steps=50,

    weight_decay=0.01,

    lr_scheduler_type="constant",

    fp16=False,
    bf16=False,

    report_to="none"
)

## Inisialisasi Trainer

Trainer digunakan untuk mengelola proses fine-tuning, evaluasi validation set, dan penyimpanan checkpoint selama pelatihan.

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    args=training_args,
)

## Fine-Tuning Model

Proses pelatihan dilakukan menggunakan dataset pelatihan yang telah diformat sebelumnya dengan evaluasi validation set setiap 200 langkah.

In [ ]:
trainer.train()

In [ ]:
# Informasi best model
print(
    "Best Checkpoint:",
    trainer.state.best_model_checkpoint
)

print(
    "Best Eval Loss:",
    trainer.state.best_metric
)

In [ ]:
best_model_info = {
    "best_checkpoint":
        trainer.state.best_model_checkpoint,

    "best_eval_loss":
        trainer.state.best_metric
}

with open(
    "best_model_info.json",
    "w"
) as f:

    json.dump(
        best_model_info,
        f,
        indent=4
    )

In [ ]:
with open(
    "training_config.json",
    "w"
) as f:

    json.dump(
        training_args.to_dict(),
        f,
        indent=4
    )

In [ ]:
# Save LoRA Adapter
trainer.save_model(
    "./mistral-usability-lora"
)

tokenizer.save_pretrained(
    "./mistral-usability-lora"
)

In [ ]:
from huggingface_hub import (create_repo,upload_folder)

repo_id = "cindyzakya/mistral-dana-usability"

create_repo(
    repo_id=repo_id,
    repo_type="model",
    exist_ok=True
)

upload_folder(
    repo_id=repo_id,
    folder_path="./mistral-usability-lora",
    commit_message="Upload final LoRA adapter"
)

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

print(
    api.model_info(
        "cindyzakya/mistral-dana-usability"
    ).id
)

In [ ]:
training_logs = pd.DataFrame(
    trainer.state.log_history
)

training_logs.to_csv(
    "training_logs.csv",
    index=False
)

training_logs.head()

In [ ]:
training_logs.tail()

In [ ]:
train_logs = training_logs[
    training_logs["loss"].notna()
]

plt.figure(figsize=(8,5))

plt.plot(
    train_logs["step"],
    train_logs["loss"]
)

plt.xlabel("Step")
plt.ylabel("Training Loss")
plt.title("Training Loss During Fine-Tuning")

plt.show()

In [ ]:
eval_logs = training_logs[
    training_logs["eval_loss"].notna()
]

plt.figure(figsize=(8,5))

plt.plot(
    eval_logs["epoch"],
    eval_logs["eval_loss"]
)

plt.xlabel("Epoch")
plt.ylabel("Validation Loss")

plt.title(
    "Validation Loss During Fine-Tuning"
)

plt.show()

In [ ]:
train_logs = training_logs[
    training_logs["loss"].notna()
]

eval_logs = training_logs[
    training_logs["eval_loss"].notna()
]

plt.figure(figsize=(8,5))

plt.plot(
    train_logs["epoch"],
    train_logs["loss"],
    label="Train Loss"
)

plt.plot(
    eval_logs["epoch"],
    eval_logs["eval_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title("Training and Validation Loss")

plt.legend()

plt.show()

In [ ]:
# Load training log
logs = pd.read_csv("/kaggle/working/training_logs.csv")

# Ambil data training loss
train_logs = logs[
    logs["loss"].notna()
][["step", "loss"]]

# Ambil data validation loss
eval_logs = logs[
    logs["eval_loss"].notna()
][["step", "eval_loss"]]

# Visualisasi
plt.figure(figsize=(8,5))

plt.plot(
    train_logs["step"],
    train_logs["loss"],
    marker="o",
    linewidth=2,
    label="Training Loss"
)

plt.plot(
    eval_logs["step"],
    eval_logs["eval_loss"],
    marker="s",
    linewidth=2,
    label="Validation Loss"
)

plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training and Validation Loss During Fine-Tuning")

plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()